# 03. 문장 임베딩과 의미 검색

문장을 **벡터(임베딩)** 로 바꾸면, 의미가 비슷한 문장끼리 벡터 공간에서 가까워집니다.
이 성질을 이용하면 키워드가 아닌 **의미 기반 검색**이 가능합니다.

1. 사전학습 모델로 문장 임베딩 추출
2. 코사인 유사도로 문장 간 의미 비교
3. 질의에 가장 가까운 문서 검색

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

checkpoint = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
print('모델 로드 완료, device:', device)

## 1. 임베딩 추출 함수

토큰별 출력을 평균(mean pooling)해 문장 하나당 벡터 하나를 얻습니다. 정규화하면 코사인 유사도 계산이 간단해집니다.

In [ ]:
def embed(sentences):
    enc = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**enc)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    summed = (out.last_hidden_state * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    emb = summed / counts                      # mean pooling
    return F.normalize(emb, p=2, dim=1)        # L2 정규화

## 2. 문장 간 의미 유사도

서로 다른 문장들의 임베딩을 구해 코사인 유사도를 비교합니다. 의미가 가까운 쌍의 점수가 높게 나옵니다.

In [ ]:
sentences = [
    'A man is playing a guitar.',
    'Someone is performing music on a string instrument.',
    'The weather is sunny today.',
]
emb = embed(sentences)
sim = emb @ emb.T   # 정규화했으므로 내적 = 코사인 유사도
print('문장 0 vs 1 (유사):', f'{sim[0,1].item():.3f}')
print('문장 0 vs 2 (무관):', f'{sim[0,2].item():.3f}')

## 3. 의미 기반 검색

여러 문서 중에서 질의와 의미가 가장 가까운 문서를 찾습니다. 키워드가 겹치지 않아도 의미로 검색됩니다.

In [ ]:
corpus = [
    'Python is a popular programming language for data science.',
    'The cat sat quietly on the warm windowsill.',
    'Deep learning models require large amounts of training data.',
    'She enjoys hiking in the mountains every weekend.',
]
query = 'What language is used for machine learning?'

corpus_emb = embed(corpus)
query_emb = embed([query])
scores = (query_emb @ corpus_emb.T).squeeze(0)

best = torch.argsort(scores, descending=True)
print('질의:', query, '\n')
for rank, i in enumerate(best, 1):
    print(f'{rank}. ({scores[i].item():.3f}) {corpus[i]}')

### 정리

- 사전학습 모델로 문장을 임베딩으로 변환하고, 코사인 유사도로 의미적 가까움을 측정했습니다.
- 의미 기반 검색은 키워드가 일치하지 않아도 뜻이 통하는 문서를 찾아냅니다.
- 이 방식은 RAG(검색 증강 생성), 추천, 중복 탐지 등의 기반 기술입니다.